# 03 - Free zone-map

Each column's class (1 byte) is already a zone-map: it bounds the possible range. A query for a value that only fits in wide columns skips the narrow ones, reading **only the class bytes**. Deterministic model (no timing noise).

In [ ]:
import matplotlib.pyplot as plt

# "Free" zone-map model: each column carries 1 class byte (range bound).
# A query for a large value skips columns whose class can't reach that value,
# reading ONLY the class bytes (metadata), without touching the payload.
NCOLS = 2000
ROWS_PER_COL = 100_000      # payload per column (elements)
class_bound = {8: 255, 16: 65535, 32: 4294967295}

def simulate(frac_wide):
    # 'frac_wide' of the columns are u32 (may hold large values); the rest, u8.
    wide = int(NCOLS * frac_wide)
    cols = [32]*wide + [8]*(NCOLS-wide)
    V = 1000          # query: "any values >= 1000?" (only u32 columns qualify)
    must_scan = [c for c in cols if class_bound[c] >= V]
    skipped = NCOLS - len(must_scan)
    meta_bytes = NCOLS                      # 1 class byte per column
    full_bytes = sum((c//8)*ROWS_PER_COL for c in cols)
    scan_bytes = sum((c//8)*ROWS_PER_COL for c in must_scan) + meta_bytes
    return skipped/NCOLS*100, (1 - scan_bytes/full_bytes)*100

fracs = [0.005,0.01,0.02,0.05,0.10,0.20]
pct_skip = []; pct_band = []
for f in fracs:
    s,b = simulate(f); pct_skip.append(s); pct_band.append(b)
    print(f"{f*100:5.1f}% wide columns -> skips {s:4.1f}% of columns, avoids {b:4.1f}% of bandwidth")

fig, ax = plt.subplots(figsize=(9,4))
ax.plot([f*100 for f in fracs], pct_skip, "o-", label="% columns skipped", color="#2E5A88", lw=2)
ax.plot([f*100 for f in fracs], pct_band, "s--", label="% bandwidth avoided", color="#2E7D5B", lw=2)
ax.set_xlabel("% of 'wide' columns (u32)"); ax.set_ylabel("%")
ax.set_title("Free zone-map: the class (1 byte/column) skips the scan")
ax.legend(); ax.grid(alpha=0.3); plt.show()

Reproduces the whitepaper order of magnitude: a few metadata bytes skip the vast majority of columns and avoid almost all the read bandwidth.